# 02 — Silver Layer: Historical and Incremental Loads

This notebook supports two execution modes:

- **Historico**: full refresh of all Silver tables.
- **Incremental**: MERGE for customers, products, and details, plus replacement of the current month and the two previous purchase partitions.


In [ ]:
dbutils.widgets.text("tipo_carga", "Incremental")
dbutils.widgets.text("fecha_carga", "2025-06-30")

load_type = dbutils.widgets.get("tipo_carga").strip()
load_date = dbutils.widgets.get("fecha_carga").strip()

if load_type.lower() not in {"historico", "incremental"}:
    raise ValueError("tipo_carga must be 'Historico' or 'Incremental'.")

print(f"Load type: {load_type}")
print(f"Load date: {load_date}")


In [ ]:
CATALOG = "sales_store"
SCHEMA = "linio"

BRONZE_PURCHASES_TABLE = f"{CATALOG}.{SCHEMA}.bronze_compras"
BRONZE_DETAILS_TABLE = f"{CATALOG}.{SCHEMA}.bronze_detalles"

SILVER_PURCHASES_TABLE = f"{CATALOG}.{SCHEMA}.silver_compras"
SILVER_CUSTOMERS_TABLE = f"{CATALOG}.{SCHEMA}.silver_clientes"
SILVER_PRODUCTS_TABLE = f"{CATALOG}.{SCHEMA}.silver_productos"
SILVER_DETAILS_TABLE = f"{CATALOG}.{SCHEMA}.silver_detalles"


In [ ]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col, concat, current_timestamp, datediff, initcap, length, lit, lpad,
    row_number, split, substring, substring_index, to_date, trim, trunc,
    upper, when
)
from pyspark.sql.window import Window


## Define the three processing partitions


In [ ]:
reference_date = datetime.strptime(load_date, "%Y-%m-%d")

partition_dates = [
    (reference_date - relativedelta(months=2)).replace(day=1),
    (reference_date - relativedelta(months=1)).replace(day=1),
    reference_date.replace(day=1)
]

partition_values = [value.strftime("%Y-%m-%d") for value in partition_dates]
replace_where = "periodo IN (" + ",".join(f"'{value}'" for value in partition_values) + ")"

print("Partitions to process:", partition_values)


## Transform purchases


In [ ]:
purchases_raw = spark.table(BRONZE_PURCHASES_TABLE)

df_purchases = (
    purchases_raw
    .withColumn("venta_id", col("venta_id").cast("integer"))
    .withColumn("estado", col("estado").cast("integer"))
    .withColumn("fecha_orden", to_date(trim(col("fecha_orden")), "yyyy-MM-dd"))
    .withColumn("fecha_entrega", to_date(trim(col("fecha_entrega")), "dd/MM/yyyy"))
    .withColumn(
        "fecha_envio",
        to_date(
            when(
                trim(col("fecha_envio")).isin("NaN", '"NaN"', ""),
                None
            ).otherwise(trim(col("fecha_envio"))),
            "dd-MM-yy"
        )
    )
    .withColumn("factura", upper(trim(col("factura"))))
    .withColumn(
        "vendedor",
        when(
            col("vendedor").isNull() | (trim(col("vendedor")) == ""),
            lit("No Identificado")
        ).otherwise(trim(col("vendedor")))
    )
    .withColumn("departamento", trim(col("departamento")))
    .withColumn("metodo_pago", trim(col("metodo_pago")))
    .withColumn("cliente_id", substring_index(col("cliente_code"), "-", 1).cast("integer"))
    .withColumn("num_documento", trim(substring_index(col("cliente_code"), "-", -1)))
    .withColumn(
        "dias_envio",
        when(col("estado") == 5, datediff(col("fecha_envio"), col("fecha_orden")))
    )
    .withColumn(
        "grupo_dias_envio",
        when(col("dias_envio").isNull(), None)
        .when(col("dias_envio") <= 3, lit("[0 - 3 días]"))
        .when(col("dias_envio") <= 7, lit("[4 - 7 días]"))
        .otherwise(lit("[más de 8 días]"))
    )
    .withColumn("periodo", trunc(col("fecha_orden"), "month"))
    .withColumn("fecha_actualizacion", current_timestamp())
)

df_silver_purchases = df_purchases.select(
    "venta_id", "factura", "tipo_compra", "fecha_orden", "fecha_entrega",
    "fecha_envio", "estado", "cliente_id", "vendedor", "departamento",
    "metodo_pago", "dias_envio", "grupo_dias_envio", "periodo",
    "fecha_actualizacion"
)


## Build customers


In [ ]:
customer_window = Window.partitionBy("cliente_id").orderBy(col("fecha_orden").desc())

df_customers = (
    df_purchases
    .withColumn("_row_number", row_number().over(customer_window))
    .filter(col("_row_number") == 1)
    .drop("_row_number")
    .select(
        col("cliente_id"),
        lpad(trim(col("num_documento")), 8, "0").alias("num_documento"),
        trim(initcap(col("nombres"))).alias("nombres"),
        trim(initcap(col("apellidos"))).alias("apellidos"),
        upper(trim(col("tipo_cliente"))).alias("tipo_cliente")
    )
    .withColumn(
        "tipo_documento",
        when(length(col("num_documento")) == 8, lit("DNI"))
        .when(
            (length(col("num_documento")) == 11) &
            (substring(col("num_documento"), 1, 2) == "10"),
            lit("RUC10")
        )
        .when(
            (length(col("num_documento")) == 11) &
            (substring(col("num_documento"), 1, 2) == "20"),
            lit("RUC20")
        )
    )
    .withColumn("nombre_completo", concat(col("nombres"), lit(" "), col("apellidos")))
    .select(
        "cliente_id", "tipo_documento", "num_documento",
        "nombre_completo", "tipo_cliente"
    )
    .withColumn("fecha_actualizacion", current_timestamp())
    .filter(col("cliente_id").isNotNull())
)


## Transform details and derive products


In [ ]:
details_raw = spark.table(BRONZE_DETAILS_TABLE)

df_details_transformed = (
    details_raw
    .withColumn("detalle_id", col("detalle_id").cast("integer"))
    .withColumn("oferta_id", col("oferta_id").cast("integer"))
    .withColumn("unidades", col("unidades").cast("integer"))
    .withColumn("precio_unitario", col("precio_unitario").cast("double"))
    .withColumn("factura", upper(trim(col("factura"))))
    .withColumn("producto", trim(col("producto")))
    .withColumn("subtotal", col("unidades") * col("precio_unitario"))
    .withColumn("tienda", split(col("nombre_archivo"), "\\.")[0])
)

product_window = Window.partitionBy("producto").orderBy("producto")

df_products = (
    df_details_transformed
    .withColumn("_row_number", row_number().over(product_window))
    .filter(col("_row_number") == 1)
    .drop("_row_number")
    .select(
        trim(col("producto")).alias("producto"),
        upper(trim(col("categoria"))).alias("categoria"),
        upper(trim(col("subcategoria"))).alias("subcategoria")
    )
    .withColumn("fecha_actualizacion", current_timestamp())
)


## Load customers and products first


In [ ]:
is_historical = load_type.lower() == "historico"

if is_historical:
    spark.sql(f"TRUNCATE TABLE {SILVER_CUSTOMERS_TABLE}")
    (
        df_customers.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_CUSTOMERS_TABLE)
    )

    spark.sql(f"TRUNCATE TABLE {SILVER_PRODUCTS_TABLE}")
    (
        df_products.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_PRODUCTS_TABLE)
    )
else:
    (
        DeltaTable.forName(spark, SILVER_CUSTOMERS_TABLE)
        .alias("target")
        .merge(df_customers.alias("source"), "target.cliente_id = source.cliente_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    (
        DeltaTable.forName(spark, SILVER_PRODUCTS_TABLE)
        .alias("target")
        .merge(df_products.alias("source"), "target.producto = source.producto")
        .whenMatchedUpdate(set={
            "categoria": "source.categoria",
            "subcategoria": "source.subcategoria",
            "fecha_actualizacion": "source.fecha_actualizacion"
        })
        .whenNotMatchedInsert(values={
            "producto": "source.producto",
            "categoria": "source.categoria",
            "subcategoria": "source.subcategoria",
            "fecha_actualizacion": "source.fecha_actualizacion"
        })
        .execute()
    )


## Build and load Silver details


In [ ]:
product_lookup = spark.table(SILVER_PRODUCTS_TABLE).select("producto_id", "producto")

df_silver_details = (
    df_details_transformed.alias("details")
    .join(product_lookup.alias("products"), "producto", "inner")
    .select(
        col("details.detalle_id").alias("detalle_id"),
        col("details.factura").alias("factura"),
        col("details.oferta_id").alias("oferta_id"),
        col("products.producto_id").alias("producto_id"),
        col("details.unidades").alias("unidades"),
        col("details.precio_unitario").alias("precio_unitario"),
        col("details.subtotal").alias("subtotal"),
        current_timestamp().alias("fecha_actualizacion")
    )
)

if is_historical:
    spark.sql(f"TRUNCATE TABLE {SILVER_DETAILS_TABLE}")
    (
        df_silver_details.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_DETAILS_TABLE)
    )
else:
    (
        DeltaTable.forName(spark, SILVER_DETAILS_TABLE)
        .alias("target")
        .merge(df_silver_details.alias("source"), "target.detalle_id = source.detalle_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


## Load Silver purchases


In [ ]:
if is_historical:
    (
        df_silver_purchases.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "false")
        .partitionBy("periodo")
        .saveAsTable(SILVER_PURCHASES_TABLE)
    )
else:
    df_incremental_purchases = df_silver_purchases.filter(
        col("periodo").isin(partition_values)
    )

    (
        df_incremental_purchases.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", replace_where)
        .saveAsTable(SILVER_PURCHASES_TABLE)
    )

print("Silver load completed.")


## Validation


In [ ]:
display(
    spark.sql(f'''
        SELECT 'silver_compras' AS table_name, COUNT(*) AS row_count FROM {SILVER_PURCHASES_TABLE}
        UNION ALL
        SELECT 'silver_clientes', COUNT(*) FROM {SILVER_CUSTOMERS_TABLE}
        UNION ALL
        SELECT 'silver_productos', COUNT(*) FROM {SILVER_PRODUCTS_TABLE}
        UNION ALL
        SELECT 'silver_detalles', COUNT(*) FROM {SILVER_DETAILS_TABLE}
    ''')
)
